# Retriever upgrade (implementing hub-aware gating + personalised travel radius)

In [2]:
import pandas as pd
import numpy as np

VISITS_PATH = r"C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step1_outputs\user_visit_events.parquet"
BUSINESS_PATH = r"C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step1_outputs\business_lookup.parquet"
MOBILITY_PATH = r"C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step2_outputs\user_mobility_table.csv"
HUBS_PATH = r"C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step2_outputs\user_hubs.csv"

# use your auto-loader if parquet is inconsistent
visits = pd.read_csv(VISITS_PATH)
businesses = pd.read_csv(BUSINESS_PATH)
mobility = pd.read_csv(MOBILITY_PATH)
hubs = pd.read_csv(HUBS_PATH)

print("visits:", visits.shape)
print("businesses:", businesses.shape)
print("mobility:", mobility.shape)
print("hubs:", hubs.shape)
display(businesses.head())

visits: (27732, 14)
businesses: (14027, 10)
mobility: (1159, 19)
hubs: (1423, 6)


,business_id,name,latitude,longitude,city,state,categories,stars,review_count,is_open
0,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,39.953949,-75.143226,Philadelphia,PA,"Sushi Bars, Restaurants, Japanese",4.0,245,1
1,ROeacJQwBeh05Rqg7F6TCg,BAP,39.943223,-75.162568,Philadelphia,PA,"Korean, Restaurants",4.5,205,1
2,9OG5YkX1g2GReZM0AskizA,Romano's Macaroni Grill,39.476117,-119.789339,Reno,NV,"Restaurants, Italian",2.5,339,1
3,QdN72BWoyFypdGJhhI5r7g,Bar One,39.939825,-75.157447,Philadelphia,PA,"Cocktail Bars, Bars, Italian, Nightlife, Resta...",4.0,65,0
4,kV_Q1oqis8Qli8dUoGpTyQ,Ardmore Pizza,40.006707,-75.289671,Ardmore,PA,"Pizza, Restaurants",3.5,109,1


In [3]:
biz_cols_needed = ["business_id", "latitude", "longitude"]
missing = [c for c in biz_cols_needed if c not in businesses.columns]
if missing:
    raise ValueError(f"businesses missing columns: {missing}")

businesses = businesses.drop_duplicates(subset=["business_id"]).copy()

# popularity from visits
biz_pop = (
    visits.groupby("business_id")
    .size()
    .reset_index(name="visit_count")
)

businesses = businesses.merge(biz_pop, on="business_id", how="left")
businesses["visit_count"] = businesses["visit_count"].fillna(0)

print("unique businesses:", businesses["business_id"].nunique())
display(businesses.head())

unique businesses: 14027


,business_id,name,latitude,longitude,city,state,categories,stars,review_count,is_open,visit_count
0,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,39.953949,-75.143226,Philadelphia,PA,"Sushi Bars, Restaurants, Japanese",4.0,245,1,2
1,ROeacJQwBeh05Rqg7F6TCg,BAP,39.943223,-75.162568,Philadelphia,PA,"Korean, Restaurants",4.5,205,1,1
2,9OG5YkX1g2GReZM0AskizA,Romano's Macaroni Grill,39.476117,-119.789339,Reno,NV,"Restaurants, Italian",2.5,339,1,3
3,QdN72BWoyFypdGJhhI5r7g,Bar One,39.939825,-75.157447,Philadelphia,PA,"Cocktail Bars, Bars, Italian, Nightlife, Resta...",4.0,65,0,2
4,kV_Q1oqis8Qli8dUoGpTyQ,Ardmore Pizza,40.006707,-75.289671,Ardmore,PA,"Pizza, Restaurants",3.5,109,1,3


In [4]:
EARTH_KM = 6371.0

def haversine_km(lat1, lon1, lat2, lon2):
    a = np.radians([lat1, lon1, lat2, lon2])
    lat1, lon1, lat2, lon2 = a
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    h = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return float(2 * EARTH_KM * np.arcsin(np.sqrt(h)))

def haversine_vec(lat1, lon1, lat2_arr, lon2_arr):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2_arr)
    lon2 = np.radians(lon2_arr)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    h = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * EARTH_KM * np.arcsin(np.sqrt(h))

In [5]:
user_visited = (
    visits.groupby("user_id")["business_id"]
    .apply(set)
    .to_dict()
)

print("users with visited sets:", len(user_visited))

users with visited sets: 7393


In [6]:
def clip_radius(x, low=3.0, high=20.0):
    return max(low, min(high, x))

def get_user_radius(row):
    label = row["mobility_label"]
    tr = row["travel_range_km"]

    if pd.isna(tr):
        tr = 5.0

    base = clip_radius(2.0 * tr, low=3.0, high=20.0)

    if label == "one_area":
        return base
    elif label == "two_area":
        return clip_radius(base * 1.2, low=3.0, high=25.0)
    elif label == "explorer":
        return clip_radius(base * 1.5, low=5.0, high=30.0)
    else:  # sparse
        return 10.0

In [7]:
def get_user_hubs(uid, mobility_row, hubs_df):
    label = mobility_row["mobility_label"]

    uh = hubs_df[hubs_df["user_id"] == uid].sort_values("hub_rank").copy()

    if label == "one_area":
        return uh.head(1)
    elif label == "two_area":
        return uh.head(2)
    elif label == "explorer":
        return uh.head(3)
    else:
        return pd.DataFrame(columns=uh.columns)

In [8]:
def retrieve_candidates_for_user(uid, businesses, mobility_df, hubs_df, user_visited, top_n=100):
    row = mobility_df[mobility_df["user_id"] == uid]
    if len(row) == 0:
        return pd.DataFrame()

    row = row.iloc[0]
    label = row["mobility_label"]
    radius_km = get_user_radius(row)

    visited = user_visited.get(uid, set())
    cand = businesses[~businesses["business_id"].isin(visited)].copy()

    selected_hubs = get_user_hubs(uid, row, hubs_df)

    # sparse fallback: global popularity
    if label == "sparse" or len(selected_hubs) == 0:
        out = cand.sort_values("visit_count", ascending=False).head(top_n).copy()
        out["retrieval_mode"] = "global_fallback"
        out["distance_km"] = np.nan
        out["radius_km"] = radius_km
        return out

    # distance to nearest selected hub
    hub_dist_cols = []
    for i, (_, h) in enumerate(selected_hubs.iterrows(), start=1):
        dcol = f"dist_hub_{i}"
        cand[dcol] = haversine_vec(
            h["hub_lat"], h["hub_lon"],
            cand["latitude"].to_numpy(),
            cand["longitude"].to_numpy()
        )
        hub_dist_cols.append(dcol)

    cand["distance_km"] = cand[hub_dist_cols].min(axis=1)

    # gate by radius
    gated = cand[cand["distance_km"] <= radius_km].copy()

    # if too few, relax once
    if len(gated) < top_n:
        gated = cand[cand["distance_km"] <= radius_km * 1.5].copy()

    # simple hybrid score: closer + more popular
    gated["pop_score"] = np.log1p(gated["visit_count"])
    gated["score"] = gated["pop_score"] - 0.1 * gated["distance_km"]

    gated = gated.sort_values(["score", "visit_count"], ascending=[False, False]).head(top_n).copy()
    gated["retrieval_mode"] = "hub_aware"
    gated["radius_km"] = radius_km
    return gated

In [9]:
sample_users = mobility.sample(8, random_state=42)["user_id"].tolist()

for uid in sample_users:
    row = mobility[mobility["user_id"] == uid].iloc[0]
    cands = retrieve_candidates_for_user(uid, businesses, mobility, hubs, user_visited, top_n=10)

    print("\nUSER:", uid)
    print("label:", row["mobility_label"])
    print("travel_range_km:", row["travel_range_km"])
    print("n_hubs:", row["n_hubs"])
    display(cands[["business_id", "visit_count", "distance_km", "radius_km", "retrieval_mode"]].head(10))


USER: uPvyXXs3XWY7iYk9Sqy-wQ
label: sparse
travel_range_km: 0.1097075397694573
n_hubs: 1


,business_id,visit_count,distance_km,radius_km,retrieval_mode
10503,ac1AeYqs8Z4_e2X5M3if2A,46,NaN,10.0,global_fallback
13378,ytynqOUb3hjKeJfRj5Tshw,45,NaN,10.0,global_fallback
10611,_ab50qdWOk0DdB6XOrBitw,39,NaN,10.0,global_fallback
8611,GXFMD0Z4jEVZBCsbPf4CTQ,39,NaN,10.0,global_fallback
10631,PP3BBaVxZLcJU54uP_wL6Q,37,NaN,10.0,global_fallback
2936,VQcCL9PiNL_wkGf-uF3fjg,35,NaN,10.0,global_fallback
9396,iSRTaT9WngzB8JJ2YKJUig,35,NaN,10.0,global_fallback
13715,oBNrLz4EDhiscSlbOl8uAw,31,NaN,10.0,global_fallback
12385,I_3LMZ_1m2mzR0oLIOePIg,30,NaN,10.0,global_fallback
12823,QHWYlmVbLC3K6eglWoHVvA,29,NaN,10.0,global_fallback



USER: CQDg6a4wDgkqRUJZSzXc-w
label: explorer
travel_range_km: 5.871318680230505
n_hubs: 2


,business_id,visit_count,distance_km,radius_km,retrieval_mode
12385,I_3LMZ_1m2mzR0oLIOePIg,30,8.340754,17.613956,hub_aware
6212,YPTYOQO8Lg9BtHsRwYBY7g,12,5.156470,17.613956,hub_aware
6770,R8t9g5nvi7VFyS8zsgmj8Q,14,7.271044,17.613956,hub_aware
4666,S26FJcC298XNpN2cZiwOrA,13,7.364161,17.613956,hub_aware
7817,Nvx2MceKWAtikcAxk_TBMg,10,5.269413,17.613956,hub_aware
11253,3Ym00dQ_oqZ0zunGMLhX3Q,7,2.259920,17.613956,hub_aware
7800,-kHHi8y4kaTI6WUVdJSdvQ,12,7.287366,17.613956,hub_aware
13958,ELUxZOuHj2MrMXV3mg3zVQ,8,3.697441,17.613956,hub_aware
12703,HFzi6UElerU4iPGXLSbwPw,11,6.886579,17.613956,hub_aware
613,vD2jzpPv4iyOLKzITscGvA,6,1.504038,17.613956,hub_aware



USER: 4OcTMOPBwY-IET2Qi6e9GA
label: one_area
travel_range_km: 3.657419748626972
n_hubs: 1


...

,business_id,visit_count,distance_km,radius_km,retrieval_mode
10503,ac1AeYqs8Z4_e2X5M3if2A,46,3.941134,7.544551,hub_aware
10611,_ab50qdWOk0DdB6XOrBitw,39,3.917872,7.544551,hub_aware
2936,VQcCL9PiNL_wkGf-uF3fjg,35,4.033463,7.544551,hub_aware
9396,iSRTaT9WngzB8JJ2YKJUig,35,4.377074,7.544551,hub_aware
13715,oBNrLz4EDhiscSlbOl8uAw,31,4.230272,7.544551,hub_aware
6092,VaO-VW3e1kARkU9bP1E7Fw,28,3.898179,7.544551,hub_aware
6868,VVH6k9-ycttH3TV_lk5WfQ,23,2.489495,7.544551,hub_aware
3204,FEXhWNCMkv22qG04E83Qjg,25,4.394544,7.544551,hub_aware
1676,DcBLYSvOuWcNReolRVr12A,25,4.768995,7.544551,hub_aware
4536,EagkHaaC-kUozD3MPzbRIw,22,4.270960,7.544551,hub_aware



USER: bzrbURQEIsn7-j3mXyoxRA
label: sparse
travel_range_km: 2.2462003207331005
n_hubs: 1


,business_id,visit_count,distance_km,radius_km,retrieval_mode
10503,ac1AeYqs8Z4_e2X5M3if2A,46,NaN,10.0,global_fallback
13378,ytynqOUb3hjKeJfRj5Tshw,45,NaN,10.0,global_fallback
8611,GXFMD0Z4jEVZBCsbPf4CTQ,39,NaN,10.0,global_fallback
10611,_ab50qdWOk0DdB6XOrBitw,39,NaN,10.0,global_fallback
10631,PP3BBaVxZLcJU54uP_wL6Q,37,NaN,10.0,global_fallback
2936,VQcCL9PiNL_wkGf-uF3fjg,35,NaN,10.0,global_fallback
9396,iSRTaT9WngzB8JJ2YKJUig,35,NaN,10.0,global_fallback
13715,oBNrLz4EDhiscSlbOl8uAw,31,NaN,10.0,global_fallback
12385,I_3LMZ_1m2mzR0oLIOePIg,30,NaN,10.0,global_fallback
12823,QHWYlmVbLC3K6eglWoHVvA,29,NaN,10.0,global_fallback



USER: TQwq95VbVCBeHHQW6vOIlw
label: one_area
travel_range_km: 3.0568011080640787
n_hubs: 1


,business_id,visit_count,distance_km,radius_km,retrieval_mode
11743,wQUBiBqlzC6cbdkX-GaBqQ,8,2.765723,6.113602,hub_aware
8334,WnVNjr9zVEpK85T7dbAfEg,8,3.415190,6.113602,hub_aware
10142,zcPCPkaTp46iJxwMv0LjVw,6,1.482207,6.113602,hub_aware
6175,mqKz51qAxfcBaUhNCGcysg,6,1.795211,6.113602,hub_aware
7246,-e1BUAOBk9RAu89c6uTyGA,7,3.386838,6.113602,hub_aware
2417,8SlHynYA-QN3UYRpKvNDgA,8,4.869266,6.113602,hub_aware
1479,Tyub4uN0yku6wyS1JWMejA,6,2.523906,6.113602,hub_aware
4339,VhBW615VRkiQLP03oAZ59Q,7,4.367138,6.113602,hub_aware
4845,BD1FU6xsYPtbQZ8pXk0_gw,5,1.706026,6.113602,hub_aware
11303,fchpEk-UJf_AUz-51LEidg,6,3.389526,6.113602,hub_aware


In [10]:
def retrieve_global_baseline(uid, businesses, user_visited, top_n=100):
    visited = user_visited.get(uid, set())
    cand = businesses[~businesses["business_id"].isin(visited)].copy()
    out = cand.sort_values("visit_count", ascending=False).head(top_n).copy()
    out["retrieval_mode"] = "global_popularity"
    out["distance_km"] = np.nan
    out["radius_km"] = np.nan
    return out

In [11]:
rows = []

for uid in mobility["user_id"]:
    row = mobility[mobility["user_id"] == uid].iloc[0]
    cands = retrieve_candidates_for_user(uid, businesses, mobility, hubs, user_visited, top_n=100)

    rows.append({
        "user_id": uid,
        "mobility_label": row["mobility_label"],
        "n_candidates": len(cands),
        "radius_km": get_user_radius(row),
        "mode": cands["retrieval_mode"].iloc[0] if len(cands) else "none"
    })

retrieval_summary = pd.DataFrame(rows)

display(
    retrieval_summary.groupby("mobility_label")[["n_candidates", "radius_km"]].median()
)

,n_candidates,radius_km
mobility_label,,
explorer,100.0,5.363604
one_area,100.0,3.533110
sparse,100.0,10.000000
two_area,100.0,3.600000


# Useful check

In [12]:
retrieval_summary["mode"].value_counts()

mode
global_fallback    585
hub_aware          574
Name: count, dtype: int64

# Printing artifacts

In [13]:
from pathlib import Path

STEP3_DIR = Path(r"C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step3_outputs")
STEP3_DIR.mkdir(parents=True, exist_ok=True)

retrieval_summary.to_csv(STEP3_DIR / "retrieval_summary.csv", index=False)

print("Saved:", STEP3_DIR / "retrieval_summary.csv")

Saved: C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step3_outputs\retrieval_summary.csv


In [14]:
all_candidate_rows = []

for uid in mobility["user_id"]:
    cands = retrieve_candidates_for_user(
        uid,
        businesses,
        mobility,
        hubs,
        user_visited,
        top_n=100
    ).copy()

    if len(cands) == 0:
        continue

    cands["user_id"] = uid

    keep_cols = [
        "user_id",
        "business_id",
        "visit_count",
        "distance_km",
        "radius_km",
        "retrieval_mode",
        "score",
    ]

    existing_cols = [c for c in keep_cols if c in cands.columns]
    all_candidate_rows.append(cands[existing_cols])

step3_candidates = pd.concat(all_candidate_rows, ignore_index=True)

print(step3_candidates.shape)
display(step3_candidates.head())

(113788, 7)


,user_id,business_id,visit_count,distance_km,radius_km,retrieval_mode,score
0,-0MIp6WKJ8QvGnYZQ5ETyg,Ob01LpWh81jAtzpYirBxpA,4,0.622673,5.0,hub_aware,1.547171
1,-0MIp6WKJ8QvGnYZQ5ETyg,1yr6foIrWy0HBXVpD5Wb3w,4,1.824246,5.0,hub_aware,1.427013
2,-0MIp6WKJ8QvGnYZQ5ETyg,7hsCR9k_GND2QoQyVkITBg,3,0.828694,5.0,hub_aware,1.303425
3,-0MIp6WKJ8QvGnYZQ5ETyg,grVoVN3n0M7IV1jRUiHWHQ,3,1.440948,5.0,hub_aware,1.242200
4,-0MIp6WKJ8QvGnYZQ5ETyg,O-6k7VRRnwxwve_Zyb-a1g,2,0.589060,5.0,hub_aware,1.039706


In [15]:
step3_candidates.to_csv(STEP3_DIR / "step3_candidates.csv", index=False)
step3_candidates.to_parquet(STEP3_DIR / "step3_candidates.parquet", index=False)

print("Saved:", STEP3_DIR / "step3_candidates.csv")
print("Saved:", STEP3_DIR / "step3_candidates.parquet")

Saved: C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step3_outputs\step3_candidates.csv
Saved: C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step3_outputs\step3_candidates.parquet


In [16]:
import json

step3_config = {
    "top_n": 100,
    "sparse_fallback_radius_km": 10.0,
    "radius_rule": "base = clip(2 * travel_range_km, low=3, high=20); two_area=1.2x; explorer=1.5x; sparse=10",
    "hub_rule": {
        "one_area": 1,
        "two_area": 2,
        "explorer": 3,
        "sparse": 0
    },
    "scoring_rule": "score = log1p(visit_count) - 0.1 * distance_km",
}

with open(STEP3_DIR / "step3_config.json", "w", encoding="utf-8") as f:
    json.dump(step3_config, f, indent=2)

print("Saved:", STEP3_DIR / "step3_config.json")

Saved: C:\Users\lebro\OneDrive - Nanyang Technological University\Github\YelpFYP\step3_outputs\step3_config.json
